# Wav2Vec2 Speech Classifier — Kaggle Training
**Datasets:** LibriSpeech (read) + AMI (spontaneous) + Casual Conversations v2 (scripted + nonscripted)

**Output:** Full FP32 ONNX + INT8 quantized ONNX

**Runtime:** Use GPU accelerator (T4/P100)

In [ ]:
## ============================================================
## CONFIG — Edit these before running
## ============================================================

# Casual Conversations v2 download links (tab-separated: filename\turl)
# Paste the full list from Meta's download page here
CCV2_LINKS = """
CCv2_frames_part_1.zip	https://PASTE_YOUR_LINK_HERE
CCv2_frames_part_2.zip	https://PASTE_YOUR_LINK_HERE
""".strip()

# How many files per class (scripted / nonscripted) from CCv2
MAX_CCV2_PER_CLASS = 5000

# Training config
EPOCHS = 10
BATCH_SIZE = 16       # 16 for T4x2 (DataParallel splits across GPUs), 8 for single GPU
LR = 1e-5
FREEZE_LAYERS = 6
PATIENCE = 3
READ_THRESHOLD = 0.65
WINDOW_SEC = 5.0
SAMPLE_RATE = 16000
WINDOW_SAMPLES = int(WINDOW_SEC * SAMPLE_RATE)  # 80,000

In [ ]:
!pip install -q transformers datasets librosa soundfile pandas scikit-learn tqdm onnx onnxruntime
!apt-get install -qq ffmpeg > /dev/null 2>&1

import os, re, sys, json, time, zipfile, subprocess, shutil, tarfile, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import Wav2Vec2Model, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

# Paths
DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}, VRAM: {props.total_memory / 1e9:.1f} GB")

## 1. Download LibriSpeech (read speech — audiobooks)

In [ ]:
LIBRI_DIR = DATA_DIR / "librispeech"
LIBRI_DIR.mkdir(parents=True, exist_ok=True)
LIBRI_MANIFEST = LIBRI_DIR / "manifest.csv"

if LIBRI_MANIFEST.exists():
    df_check = pd.read_csv(LIBRI_MANIFEST)
    existing = df_check["filepath"].apply(os.path.exists).sum()
    if existing > 100:
        print(f"LibriSpeech already done ({existing} files). Skipping.")
    else:
        LIBRI_MANIFEST.unlink()

if not LIBRI_MANIFEST.exists():
    LIBRI_URL = "https://www.openslr.org/resources/12/train-clean-100.tar.gz"
    tar_path = LIBRI_DIR / "train-clean-100.tar.gz"
    audio_root = LIBRI_DIR / "LibriSpeech" / "train-clean-100"

    # Download
    if not tar_path.exists() and not audio_root.exists():
        print("Downloading LibriSpeech train-clean-100 (~6.3 GB)...")
        !wget -q --show-progress -O {tar_path} {LIBRI_URL}

    # Extract
    if not audio_root.exists() and tar_path.exists():
        print("Extracting...")
        with tarfile.open(str(tar_path), "r:gz") as tar:
            tar.extractall(str(LIBRI_DIR))
        tar_path.unlink()
        print("Extracted and removed tar.gz")

    # Build manifest
    rows = []
    for trans_file in sorted(audio_root.rglob("*.trans.txt")):
        with open(trans_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(" ", 1)
                if len(parts) < 2:
                    continue
                utt_id, text = parts
                flac_path = trans_file.parent / f"{utt_id}.flac"
                if flac_path.exists():
                    rows.append({
                        "filepath": str(flac_path.resolve()),
                        "filename": flac_path.name,
                        "source": "librispeech",
                        "label": "read",
                        "label_int": 1,
                        "speaker_id": utt_id.split("-")[0],
                    })

    df_libri = pd.DataFrame(rows)
    df_libri.to_csv(LIBRI_MANIFEST, index=False)
    print(f"LibriSpeech: {len(df_libri)} files")
else:
    df_libri = pd.read_csv(LIBRI_MANIFEST)
    print(f"LibriSpeech: {len(df_libri)} files (cached)")

## 2. Download AMI (spontaneous speech — meetings)

In [ ]:
from datasets import load_dataset, Audio

AMI_DIR = DATA_DIR / "ami"
AMI_DIR.mkdir(parents=True, exist_ok=True)
AMI_AUDIO = AMI_DIR / "audio"
AMI_AUDIO.mkdir(exist_ok=True)
AMI_MANIFEST = AMI_DIR / "manifest.csv"

if AMI_MANIFEST.exists():
    df_check = pd.read_csv(AMI_MANIFEST)
    existing = df_check["filepath"].apply(os.path.exists).sum()
    if existing > 100:
        print(f"AMI already done ({existing} files). Skipping.")

if not AMI_MANIFEST.exists() or existing <= 100:
    print("Downloading AMI corpus (ihm split)...")
    ds = load_dataset("edinburghcstr/ami", "ihm", split="train", trust_remote_code=True)
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    rows = []
    for i in tqdm(range(len(ds)), desc="Processing AMI"):
        try:
            item = ds[i]
            audio = item["audio"]["array"]
            duration = len(audio) / SAMPLE_RATE

            if duration < 3.0 or duration > 60.0:
                continue

            fname = f"ami_{i:06d}.wav"
            fpath = AMI_AUDIO / fname
            if not fpath.exists():
                sf.write(str(fpath), audio.astype(np.float32), SAMPLE_RATE)

            rows.append({
                "filepath": str(fpath.resolve()),
                "filename": fname,
                "source": "ami",
                "label": "spontaneous",
                "label_int": 0,
                "duration_sec": round(duration, 2),
                "speaker_id": str(item.get("speaker_id", "")),
            })
        except Exception as e:
            if i < 5:
                print(f"  Skipping {i}: {e}")
            continue

    df_ami = pd.DataFrame(rows)
    df_ami.to_csv(AMI_MANIFEST, index=False)
    print(f"AMI: {len(df_ami)} files")

    # Free memory
    del ds
    import gc; gc.collect()
else:
    df_ami = pd.read_csv(AMI_MANIFEST)
    print(f"AMI: {len(df_ami)} files (cached)")

## 3. Download Casual Conversations v2 (scripted + nonscripted)
Downloads one zip at a time, extracts only English MP4s, converts to WAV, deletes zip + MP4s to save disk.

Filename pattern inside zips: `{participant_id}_{language}_{scripted|nonscripted}_{index}.mp4`

In [ ]:
CCV2_DIR = DATA_DIR / "casual_conversations"
CCV2_DIR.mkdir(parents=True, exist_ok=True)
CCV2_AUDIO_SCRIPTED = CCV2_DIR / "audio_scripted"
CCV2_AUDIO_NONSCRIPTED = CCV2_DIR / "audio_nonscripted"
CCV2_AUDIO_SCRIPTED.mkdir(exist_ok=True)
CCV2_AUDIO_NONSCRIPTED.mkdir(exist_ok=True)
CCV2_MP4_TEMP = CCV2_DIR / "mp4_temp"
CCV2_MP4_TEMP.mkdir(exist_ok=True)
CCV2_MANIFEST = CCV2_DIR / "manifest.csv"

ENGLISH_PATTERN = re.compile(r"(\d+)_english_(scripted|nonscripted)_(\d+)\.mp4$", re.IGNORECASE)


def parse_ccv2_links(links_text):
    """Parse tab-separated links text into {filename: url} dict."""
    links = {}
    for line in links_text.strip().split("\n"):
        line = line.strip()
        if not line or line.startswith("file_name"):
            continue
        parts = line.split("\t")
        if len(parts) >= 2:
            fname, url = parts[0].strip(), parts[1].strip()
            if fname.endswith(".zip"):
                links[fname] = url
    return links


def extract_english_from_zip(zip_path, mp4_out_dir):
    """Extract only English scripted/nonscripted MP4s from a zip."""
    extracted = []
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            all_names = zf.namelist()
            english_files = [n for n in all_names if ENGLISH_PATTERN.search(n.split("/")[-1])]
            if not english_files:
                print(f"  No English MP4s in {zip_path.name}")
                return extracted
            print(f"  {len(english_files)} English files in {zip_path.name}")
            for name in tqdm(english_files, desc=f"  Extracting", leave=False):
                basename = Path(name).name
                out_path = mp4_out_dir / basename
                if not out_path.exists():
                    data = zf.read(name)
                    with open(out_path, "wb") as f:
                        f.write(data)
                extracted.append(out_path)
    except zipfile.BadZipFile:
        print(f"  ERROR: {zip_path.name} is corrupted")
    except Exception as e:
        print(f"  ERROR: {e}")
    return extracted


def mp4_to_wav(mp4_path, wav_path):
    """Extract 16kHz mono audio from MP4."""
    try:
        result = subprocess.run(
            ["ffmpeg", "-i", str(mp4_path), "-vn", "-acodec", "pcm_s16le",
             "-ar", str(SAMPLE_RATE), "-ac", "1", "-y", "-loglevel", "error", str(wav_path)],
            capture_output=True, timeout=120)
        return result.returncode == 0
    except Exception:
        return False


def convert_mp4s_to_wav(mp4_dir):
    """Convert extracted English MP4s to WAV, return manifest rows."""
    mp4_files = sorted(Path(mp4_dir).glob("*.mp4"))
    rows = []
    for mp4 in tqdm(mp4_files, desc="MP4 -> WAV"):
        match = ENGLISH_PATTERN.match(mp4.name)
        if not match:
            continue

        participant_id = match.group(1)
        script_type = match.group(2)

        # Check class limits
        scripted_so_far = len(list(CCV2_AUDIO_SCRIPTED.glob("*.wav")))
        nonscripted_so_far = len(list(CCV2_AUDIO_NONSCRIPTED.glob("*.wav")))
        if script_type == "scripted" and scripted_so_far >= MAX_CCV2_PER_CLASS:
            continue
        if script_type == "nonscripted" and nonscripted_so_far >= MAX_CCV2_PER_CLASS:
            continue

        wav_dir = CCV2_AUDIO_SCRIPTED if script_type == "scripted" else CCV2_AUDIO_NONSCRIPTED
        label = "read" if script_type == "scripted" else "spontaneous"
        label_int = 1 if script_type == "scripted" else 0

        wav_path = wav_dir / (mp4.stem + ".wav")
        if not wav_path.exists():
            if not mp4_to_wav(mp4, wav_path):
                continue

        # Check duration
        try:
            info = sf.info(str(wav_path))
            duration = info.duration
        except Exception:
            duration = 0

        if duration < 3.0 or duration > 120.0:
            wav_path.unlink(missing_ok=True)
            continue

        rows.append({
            "filepath": str(wav_path.resolve()),
            "filename": wav_path.name,
            "source": "casual_conversations",
            "label": label,
            "label_int": label_int,
            "script_type": script_type,
            "duration_sec": round(duration, 2),
            "speaker_id": participant_id,
        })
    return rows


print("Casual Conversations v2 helper functions ready.")

In [ ]:
## Download + process CCv2 zips one at a time
if CCV2_MANIFEST.exists():
    df_check = pd.read_csv(CCV2_MANIFEST)
    existing = df_check["filepath"].apply(os.path.exists).sum()
    if existing > 100:
        print(f"CCv2 already done ({existing} files). Skipping download.")
        df_ccv2 = df_check
        ccv2_skip = True
    else:
        CCV2_MANIFEST.unlink()
        ccv2_skip = False
else:
    ccv2_skip = False

if not ccv2_skip:
    links = parse_ccv2_links(CCV2_LINKS)
    if not links:
        print("WARNING: No CCv2 links found. Paste your links in CCV2_LINKS at the top.")
    else:
        # Sort by part number
        def part_num(name):
            m = re.search(r"part_(\d+)", name)
            return int(m.group(1)) if m else 0
        sorted_links = sorted(links.items(), key=lambda x: part_num(x[0]))

        print(f"Found {len(sorted_links)} CCv2 zip links")
        all_ccv2_rows = []

        for i, (fname, url) in enumerate(sorted_links):
            # Check if we have enough
            scripted_count = len(list(CCV2_AUDIO_SCRIPTED.glob("*.wav")))
            nonscripted_count = len(list(CCV2_AUDIO_NONSCRIPTED.glob("*.wav")))
            if scripted_count >= MAX_CCV2_PER_CLASS and nonscripted_count >= MAX_CCV2_PER_CLASS:
                print(f"\nReached target: {scripted_count} scripted, {nonscripted_count} nonscripted. Done.")
                break

            zip_path = CCV2_DIR / fname
            print(f"\n[{i+1}/{len(sorted_links)}] {fname}")
            print(f"  Current: {scripted_count} scripted, {nonscripted_count} nonscripted")

            # Download
            if not zip_path.exists():
                print(f"  Downloading...")
                !wget -q --show-progress -O {zip_path} "{url}"

            if not zip_path.exists() or zip_path.stat().st_size < 1000:
                print(f"  Download failed. Skipping.")
                if zip_path.exists():
                    zip_path.unlink()
                continue

            # Extract English MP4s
            extract_english_from_zip(zip_path, CCV2_MP4_TEMP)

            # Convert MP4s to WAV
            new_rows = convert_mp4s_to_wav(CCV2_MP4_TEMP)
            all_ccv2_rows.extend(new_rows)

            # Delete zip to save disk
            print(f"  Deleting {fname}...")
            zip_path.unlink(missing_ok=True)

            # Delete processed MP4s
            for f in CCV2_MP4_TEMP.glob("*.mp4"):
                f.unlink()

        # Build manifest from all WAV files on disk (in case of restart)
        final_rows = []
        for wav_dir, script_type, label, label_int in [
            (CCV2_AUDIO_SCRIPTED, "scripted", "read", 1),
            (CCV2_AUDIO_NONSCRIPTED, "nonscripted", "spontaneous", 0),
        ]:
            for wav in sorted(wav_dir.glob("*.wav")):
                match = re.match(r"(\d+)_english_", wav.name)
                pid = match.group(1) if match else ""
                try:
                    dur = sf.info(str(wav)).duration
                except Exception:
                    dur = 0
                final_rows.append({
                    "filepath": str(wav.resolve()),
                    "filename": wav.name,
                    "source": "casual_conversations",
                    "label": label,
                    "label_int": label_int,
                    "script_type": script_type,
                    "duration_sec": round(dur, 2),
                    "speaker_id": pid,
                })

        df_ccv2 = pd.DataFrame(final_rows)
        df_ccv2.to_csv(CCV2_MANIFEST, index=False)

    # Cleanup temp dir
    shutil.rmtree(CCV2_MP4_TEMP, ignore_errors=True)

scripted = df_ccv2[df_ccv2["label_int"] == 1] if len(df_ccv2) > 0 else pd.DataFrame()
nonscripted = df_ccv2[df_ccv2["label_int"] == 0] if len(df_ccv2) > 0 else pd.DataFrame()
print(f"\nCCv2 total: {len(df_ccv2)} ({len(scripted)} scripted, {len(nonscripted)} nonscripted)")

## 4. Build Balanced Unified Manifest
**Scripted (read):** LibriSpeech + CCv2 scripted
**Spontaneous:** AMI + CCv2 nonscripted

Balance: cap LibriSpeech so total read ~ total spontaneous

In [ ]:
# Reload manifests (handles restarts)
df_libri = pd.read_csv(LIBRI_DIR / "manifest.csv")
df_ami = pd.read_csv(AMI_DIR / "manifest.csv")
df_ccv2 = pd.read_csv(CCV2_MANIFEST) if CCV2_MANIFEST.exists() else pd.DataFrame()

# Filter to existing files
df_libri = df_libri[df_libri["filepath"].apply(os.path.exists)].reset_index(drop=True)
df_ami = df_ami[df_ami["filepath"].apply(os.path.exists)].reset_index(drop=True)
if len(df_ccv2) > 0:
    df_ccv2 = df_ccv2[df_ccv2["filepath"].apply(os.path.exists)].reset_index(drop=True)


def estimate_windows(df):
    """Estimate total 5-sec windows from a dataframe of audio files."""
    total = 0
    for _, row in df.iterrows():
        fp = row["filepath"]
        try:
            dur = librosa.get_duration(path=fp)
            total += max(1, int(dur * SAMPLE_RATE) // WINDOW_SAMPLES)
        except Exception:
            total += 1  # assume at least 1 window
    return total


# Standardize columns
cols = ["filepath", "filename", "source", "label", "label_int"]

libri = df_libri[cols].copy()
ami = df_ami[cols].copy()

ccv2_scripted = pd.DataFrame()
ccv2_nonscripted = pd.DataFrame()
if len(df_ccv2) > 0:
    ccv2_scripted = df_ccv2[df_ccv2["label_int"] == 1][cols].copy()
    ccv2_scripted["source"] = "ccv2_scripted"
    ccv2_nonscripted = df_ccv2[df_ccv2["label_int"] == 0][cols].copy()
    ccv2_nonscripted["source"] = "ccv2_nonscripted"

# Balance CCv2: equal scripted and nonscripted
n_ccv2_each = min(len(ccv2_scripted), len(ccv2_nonscripted))
if n_ccv2_each > 0:
    ccv2_scripted = ccv2_scripted.sample(n=n_ccv2_each, random_state=42)
    ccv2_nonscripted = ccv2_nonscripted.sample(n=n_ccv2_each, random_state=42)

# ── Window-aware balancing ──────────────────────────────────
# Count windows for each spontaneous source
print("Estimating window counts per source...")
w_ami = estimate_windows(ami)
w_ccv2_nonscr = estimate_windows(ccv2_nonscripted) if len(ccv2_nonscripted) > 0 else 0
w_ccv2_scr = estimate_windows(ccv2_scripted) if len(ccv2_scripted) > 0 else 0

total_spont_windows = w_ami + w_ccv2_nonscr
target_read_windows = total_spont_windows
read_windows_without_libri = w_ccv2_scr
target_libri_windows = max(0, target_read_windows - read_windows_without_libri)

print(f"  AMI windows:             {w_ami}")
print(f"  CCv2 nonscripted windows:{w_ccv2_nonscr}")
print(f"  CCv2 scripted windows:   {w_ccv2_scr}")
print(f"  Total spontaneous:       {total_spont_windows}")
print(f"  Target LibriSpeech windows: {target_libri_windows}")

# Estimate windows per LibriSpeech file (~avg), then cap file count
if len(libri) > 0 and target_libri_windows > 0:
    w_libri_full = estimate_windows(libri)
    avg_w_per_file = w_libri_full / len(libri) if len(libri) > 0 else 1
    target_libri_files = int(target_libri_windows / avg_w_per_file)
    target_libri_files = min(target_libri_files, len(libri))
    if target_libri_files < len(libri):
        libri = libri.sample(n=target_libri_files, random_state=42)
        w_libri = estimate_windows(libri)
        print(f"  Capped LibriSpeech to {target_libri_files} files (~{w_libri} windows)")
    else:
        w_libri = w_libri_full
        print(f"  Using all {len(libri)} LibriSpeech files (~{w_libri} windows)")
elif len(libri) > 0:
    cap = min(5000, len(libri))
    libri = libri.sample(n=cap, random_state=42)
    w_libri = estimate_windows(libri)
    print(f"  Capped LibriSpeech to {cap} files (~{w_libri} windows)")
else:
    w_libri = 0

total_read_windows = w_libri + w_ccv2_scr
print(f"\n  Final read windows:        {total_read_windows}")
print(f"  Final spontaneous windows: {total_spont_windows}")
print(f"  Ratio:                     {total_read_windows/max(total_spont_windows,1):.2f}")

# Combine all
all_dfs = [df for df in [libri, ami, ccv2_scripted, ccv2_nonscripted] if len(df) > 0]
combined = pd.concat(all_dfs, ignore_index=True)

total_read = (combined["label_int"] == 1).sum()
total_spont = (combined["label_int"] == 0).sum()

print(f"\n{'='*50}")
print(f"COMBINED DATASET (files)")
print(f"{'='*50}")
print(f"Total:       {len(combined)}")
print(f"  Read:        {total_read}")
print(f"  Spontaneous: {total_spont}")
print(f"\nBy source:")
for src in combined["source"].unique():
    sub = combined[combined["source"] == src]
    print(f"  {src:<20s}: {len(sub):>6d} ({(sub.label_int==1).sum()} read, {(sub.label_int==0).sum()} spont)")

# Stratified train/val/test split
train_df, test_df = train_test_split(combined, test_size=0.15, random_state=42, stratify=combined["label_int"])
train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=42, stratify=train_df["label_int"])

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"
manifest = pd.concat([train_df, val_df, test_df], ignore_index=True)

MANIFEST_PATH = DATA_DIR / "manifest_unified.csv"
manifest.to_csv(MANIFEST_PATH, index=False)
print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Saved: {MANIFEST_PATH}")

## 5. Dataset + Model Definition

In [ ]:
class CombinedWindowDataset(Dataset):
    """Loads audio from manifest, creates 5-sec non-overlapping windows."""

    def __init__(self, manifest_df):
        self.sr = SAMPLE_RATE
        self.window = WINDOW_SAMPLES
        self.df = manifest_df.reset_index(drop=True)
        self.window_index = []

        print("Building window index...")
        skipped = 0
        for idx, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Indexing"):
            fp = row["filepath"]
            if not os.path.exists(fp):
                skipped += 1
                continue
            try:
                duration = librosa.get_duration(path=fp)
                n_samples = int(duration * self.sr)
                n_windows = max(1, n_samples // self.window)
                for w in range(n_windows):
                    self.window_index.append((idx, w * self.window))
            except Exception:
                skipped += 1
                continue

        print(f"  {len(self.window_index)} windows from {len(self.df)} files (skipped {skipped})")

    def __len__(self):
        return len(self.window_index)

    def __getitem__(self, i):
        file_idx, start_sample = self.window_index[i]
        row = self.df.iloc[file_idx]

        offset_sec = start_sample / self.sr
        audio, _ = librosa.load(row["filepath"], sr=self.sr, mono=True,
                                offset=offset_sec, duration=WINDOW_SEC)

        if len(audio) < self.window:
            audio = np.pad(audio, (0, self.window - len(audio)))
        else:
            audio = audio[:self.window]

        return {
            "input_values": torch.tensor(audio, dtype=torch.float32),
            "labels": torch.tensor(row["label_int"], dtype=torch.long),
        }


class BiasedSpeechClassifier(nn.Module):
    """wav2vec2 encoder + classification head. Output: single logit -> P(read)."""

    def __init__(self, model_name="facebook/wav2vec2-base", hidden_size=256,
                 dropout=0.3, freeze_layers=6):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(model_name)
        encoder_dim = self.encoder.config.hidden_size  # 768

        self.encoder.feature_extractor._freeze_parameters()

        if freeze_layers > 0:
            for i, layer in enumerate(self.encoder.encoder.layers):
                if i < freeze_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(encoder_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"Model: {model_name}")
        print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
        print(f"  Frozen layers: {freeze_layers}, Hidden: {hidden_size}")

    def forward(self, input_values, attention_mask=None):
        outputs = self.encoder(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        if attention_mask is not None:
            mask = self._get_feature_vector_attention_mask(
                hidden_states.shape[1], attention_mask)
            hidden_states = hidden_states * mask.unsqueeze(-1)
            pooled = hidden_states.sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1)
        else:
            pooled = hidden_states.mean(dim=1)

        return self.classifier(pooled).squeeze(-1)

    def _get_feature_vector_attention_mask(self, feature_vector_length, attention_mask):
        output_lengths = self.encoder._get_feat_extract_output_lengths(
            attention_mask.sum(-1)).to(torch.long)
        batch_size = attention_mask.shape[0]
        mask = torch.zeros(batch_size, feature_vector_length,
                           dtype=attention_mask.dtype, device=attention_mask.device)
        for i in range(batch_size):
            mask[i, :output_lengths[i]] = 1
        return mask


def collate_fn(batch):
    return {
        "input_values": torch.stack([b["input_values"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }

print("Dataset + Model classes defined.")

## 6. Train

In [ ]:
# Load manifest
manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest[manifest["filepath"].apply(os.path.exists)].reset_index(drop=True)
manifest = manifest[manifest["label_int"].isin([0, 1])].reset_index(drop=True)

print(f"Dataset: {len(manifest)} files")
print(f"  Read:        {(manifest.label_int==1).sum()}")
print(f"  Spontaneous: {(manifest.label_int==0).sum()}")

# Use pre-made splits
train_df = manifest[manifest["split"] == "train"]
val_df = manifest[manifest["split"] == "val"]
test_df = manifest[manifest["split"] == "test"]
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Build window datasets
train_dataset = CombinedWindowDataset(train_df)
val_dataset = CombinedWindowDataset(val_df)
test_dataset = CombinedWindowDataset(test_df)

num_workers = min(4, os.cpu_count() or 2)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)

print(f"\nTrain windows: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# Build model
print(f"Building model (freeze_layers={FREEZE_LAYERS})...")
model = BiasedSpeechClassifier(freeze_layers=FREEZE_LAYERS).to(device)

# Use both GPUs if available
n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    print(f"Using DataParallel across {n_gpus} GPUs: {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")
    model = nn.DataParallel(model)

# Loss with class balancing
raw_model = model.module if isinstance(model, nn.DataParallel) else model
train_labels = [train_dataset.df.iloc[fi]["label_int"] for fi, _ in train_dataset.window_index]
n_spont = sum(1 for l in train_labels if l == 0)
n_read = sum(1 for l in train_labels if l == 1)
pos_weight = torch.tensor([n_spont / max(n_read, 1)], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Class balance: {n_spont} spont, {n_read} read, pos_weight={pos_weight.item():.3f}")

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
num_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_steps * 0.1), num_steps)
scaler = torch.cuda.amp.GradScaler()

best_f1 = 0.0
patience_counter = 0
history = []

print(f"\n{'='*60}")
print(f"TRAINING: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}, GPUs={n_gpus}")
print(f"{'='*60}")

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    print(f"\nEpoch {epoch}/{EPOCHS}")

    # Train
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for batch in tqdm(train_loader, desc="Training"):
        x = batch["input_values"].to(device)
        y = batch["labels"].to(device).float()
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(logits).detach().cpu().numpy() >= READ_THRESHOLD).astype(int)
        all_preds.extend(preds)
        all_labels.extend(y.cpu().numpy().astype(int))

    train_loss = total_loss / len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    train_f1 = f1_score(all_labels, all_preds, zero_division=0)

    # Validate
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            x = batch["input_values"].to(device)
            y = batch["labels"].to(device).float()
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss = criterion(logits, y)
            total_loss += loss.item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend((probs >= READ_THRESHOLD).astype(int))
            all_labels.extend(y.cpu().numpy().astype(int))
            all_probs.extend(probs)

    val_loss = total_loss / len(val_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, zero_division=0)

    elapsed = time.time() - epoch_start
    print(f"  Train — Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
    print(f"  Val   — Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
    print(f"  Time: {elapsed/60:.1f} min")

    history.append({
        "epoch": epoch, "train_loss": round(train_loss, 4),
        "train_f1": round(train_f1, 4), "val_loss": round(val_loss, 4),
        "val_f1": round(val_f1, 4), "time_min": round(elapsed/60, 1),
    })

    with open(CKPT_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    # Save unwrapped model (without DataParallel wrapper)
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": raw_model.state_dict(),
            "val_f1": val_f1,
            "read_threshold": READ_THRESHOLD,
        }, CKPT_DIR / "wav2vec2_best.pt")
        print(f"  -> New best (F1={val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

total_time = (time.time() - start_time) / 60
print(f"\nTotal training time: {total_time:.1f} min")

## 7. Final Test Evaluation

In [ ]:
# Load best model
ck = torch.load(CKPT_DIR / "wav2vec2_best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck["model_state_dict"])
model.eval()
print(f"Loaded best model (epoch {ck['epoch']}, val_f1={ck['val_f1']:.4f})")

# Test evaluation
total_loss = 0
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        x = batch["input_values"].to(device)
        y = batch["labels"].to(device).float()
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.item()
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend((probs >= READ_THRESHOLD).astype(int))
        all_labels.extend(y.cpu().numpy().astype(int))
        all_probs.extend(probs)

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, zero_division=0)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print(f"\nTest — Acc: {test_acc:.4f}, F1: {test_f1:.4f}")
print(classification_report(all_labels, all_preds, target_names=["spontaneous", "read"]))

# Threshold sweep
print("Threshold Sensitivity:")
for t in [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    t_preds = (all_probs >= t).astype(int)
    t_f1 = f1_score(all_labels, t_preds, zero_division=0)
    marker = " <-- default" if t == READ_THRESHOLD else ""
    print(f"  threshold={t:.2f}: f1={t_f1:.4f}{marker}")

# Save results
results = {
    "test_acc": round(test_acc, 4), "test_f1": round(test_f1, 4),
    "best_val_f1": round(best_f1, 4), "best_epoch": ck["epoch"],
    "read_threshold": READ_THRESHOLD,
}
with open(CKPT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {CKPT_DIR / 'results.json'}")

## 8. Export ONNX (FP32 + INT8 Quantized)

In [ ]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

# Build clean model for export (no dropout, no frozen layers)
class BiasedSpeechClassifierExport(nn.Module):
    def __init__(self, model_name="facebook/wav2vec2-base", hidden_size=256):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(model_name)
        encoder_dim = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(encoder_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, input_values):
        outputs = self.encoder(input_values).last_hidden_state
        pooled = outputs.mean(dim=1)
        return self.classifier(pooled).squeeze(-1)

# Load weights into export model
export_model = BiasedSpeechClassifierExport().cpu()
ck = torch.load(CKPT_DIR / "wav2vec2_best.pt", map_location="cpu", weights_only=False)
export_model.load_state_dict(ck["model_state_dict"])
export_model.eval()

# Export FP32 ONNX
onnx_path = CKPT_DIR / "wav2vec2_trained.onnx"
dummy = torch.zeros(1, WINDOW_SAMPLES)

print(f"Exporting FP32 ONNX...")
torch.onnx.export(
    export_model, dummy, str(onnx_path),
    opset_version=14,
    input_names=["input_values"],
    output_names=["logit"],
    dynamic_axes={
        "input_values": {0: "batch_size"},
        "logit": {0: "batch_size"},
    },
    do_constant_folding=True,
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
fp32_mb = onnx_path.stat().st_size / 1e6
print(f"  FP32: {fp32_mb:.1f} MB")

# Quantize to INT8
quant_path = CKPT_DIR / "wav2vec2_trained_quant.onnx"
print(f"Quantizing to INT8...")
quantize_dynamic(
    str(onnx_path), str(quant_path),
    weight_type=QuantType.QInt8,
    op_types_to_quantize=["MatMul", "Gemm"],
)
int8_mb = quant_path.stat().st_size / 1e6
print(f"  INT8: {int8_mb:.1f} MB ({100*int8_mb/fp32_mb:.0f}% of FP32)")

# Verify inference
sess = ort.InferenceSession(str(quant_path), providers=["CPUExecutionProvider"])
test_input = np.random.randn(1, WINDOW_SAMPLES).astype(np.float32)
output = sess.run(["logit"], {"input_values": test_input})
print(f"  Verification: output shape={output[0].shape}, value={output[0][0]:.4f}")

print(f"\nFiles ready for download:")
print(f"  {onnx_path.name} — FP32 ({fp32_mb:.0f} MB)")
print(f"  {quant_path.name} — INT8 ({int8_mb:.0f} MB)")
print(f"  wav2vec2_best.pt — PyTorch checkpoint")
print(f"  results.json — test metrics")
print(f"  history.json — training history")

## 9. Download Files
Run this cell, then use the Kaggle file browser (left panel) to download from `/kaggle/working/checkpoints/`

In [ ]:
# List all output files with sizes
print("Output files in checkpoints/:")
for f in sorted(CKPT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<40s} {size_mb:>8.1f} MB")

# Also copy to /kaggle/working root for easy download
shutil.copy2(CKPT_DIR / "wav2vec2_trained.onnx", "/kaggle/working/wav2vec2_trained.onnx")
shutil.copy2(CKPT_DIR / "wav2vec2_trained_quant.onnx", "/kaggle/working/wav2vec2_trained_quant.onnx")
shutil.copy2(CKPT_DIR / "wav2vec2_best.pt", "/kaggle/working/wav2vec2_best.pt")
shutil.copy2(CKPT_DIR / "results.json", "/kaggle/working/results.json")
shutil.copy2(CKPT_DIR / "history.json", "/kaggle/working/history.json")

print("\nCopied to /kaggle/working/ for easy download.")
print("Use Kaggle 'Save Version' -> 'Save & Run All' to keep output as a dataset.")